# 1일차 6교시 · 데이터 편향과 강건성

1. **그룹별 성능 비교**: 전체 성능은 좋아도 데이터가 적은 그룹에서 성능이 떨어지는지 확인합니다.
2. **섭동 테스트**: 입력에 잡음을 넣으면 성능이 얼마나 떨어지는지 봅니다.
3. **결측 강건성**: 값이 빠지면 성능이 얼마나 떨어지는지 봅니다.
4. **방향성 테스트**: 예측이 정해진 방향으로 바뀌는지 확인합니다.

그래프의 글자는 Colab에서 한글 폰트가 깨지지 않도록 영어로 적었습니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, recall_score

## 1. 그룹별 성능 비교 (가상 데이터)

- 그룹 A는 1,800명, 그룹 B는 200명입니다(대표성 부족).
- 두 그룹은 양성 비율이 비슷하지만(약 30%), **특성과 결과의 관계가 조금 다릅니다.** 그룹 B에서는 두 번째 특성이 결과에 거의 영향을 주지 않습니다.
- 모델은 그룹 정보를 모른 채 학습합니다. 데이터 대부분이 그룹 A라서, 모델은 그룹 A의 규칙을 배웁니다.

In [ ]:
rng = np.random.RandomState(0)

def make_group(n, w1, rng):
    X = rng.normal(size=(n, 6))
    score = X[:, 0] + w1 * X[:, 1] + 0.5 * X[:, 2] + rng.normal(scale=0.5, size=n)
    return X, (score > 0.8).astype(int)

XA, yA = make_group(1800, 1.0, rng)     # 그룹 A: 두 번째 특성이 중요
XB, yB = make_group(200, -0.2, rng)     # 그룹 B: 두 번째 특성의 영향이 거의 없음
X = np.vstack([XA, XB])
y = np.concatenate([yA, yB])
group = np.array(["A"] * len(yA) + ["B"] * len(yB))

X_tr, X_te, y_tr, y_te, g_tr, g_te = train_test_split(
    X, y, group, test_size=0.3, random_state=42, stratify=group)
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr, y_tr)
pred = clf.predict(X_te)

rows = [{"그룹": "전체", "테스트 건수": len(y_te),
         "정확도": round(accuracy_score(y_te, pred), 3), "재현율": round(recall_score(y_te, pred), 3)}]
for g in ["A", "B"]:
    m = g_te == g
    rows.append({"그룹": g, "테스트 건수": int(m.sum()),
                 "정확도": round(accuracy_score(y_te[m], pred[m]), 3),
                 "재현율": round(recall_score(y_te[m], pred[m]), 3)})
pd.DataFrame(rows)

- **전체 성능만 보면 그룹 B의 문제가 가려집니다.**
- 검수 기준 예: "그룹 간 재현율 차이 0.05 이하". 이 기준을 넘으면 데이터 보강이나 그룹별 분석이 필요합니다.

## 2. 섭동 테스트: 잡음에 대한 강건성

유방암 모델의 테스트 입력에 **특성별 표준편차 × 잡음 크기**만큼 무작위 잡음을 더합니다. 잡음마다 10번 반복해 평균을 냅니다.

In [ ]:
Xb, yb = load_breast_cancer(return_X_y=True)
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.25, random_state=42, stratify=yb)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xb_tr, yb_tr)
std = Xb_tr.std(axis=0)

rng = np.random.RandomState(0)
levels = [0, 0.05, 0.1, 0.2, 0.5, 1.0]
acc = []
for level in levels:
    scores = [model.score(Xb_te + rng.normal(size=Xb_te.shape) * std * level, yb_te) for _ in range(10)]
    acc.append(np.mean(scores))
perturb = pd.DataFrame({"잡음 크기(표준편차 배수)": levels, "평균 정확도": np.round(acc, 3)})
perturb

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(levels, acc, marker="o")
plt.xlabel("noise level (x feature std)")
plt.ylabel("accuracy")
plt.title("Perturbation test")
plt.show()

검수 기준 예: "표준편차의 5% 잡음에서 정확도 하락 0.02 이하". 기준을 먼저 정하고 결과와 비교합니다.

## 3. 결측 강건성

테스트 입력의 일부 값을 지운 뒤, **학습 데이터의 중앙값**으로 채워서 예측합니다.

In [ ]:
imputer = SimpleImputer(strategy="median").fit(Xb_tr)   # 학습 데이터로만 fit
rows = []
for rate in [0, 0.1, 0.3, 0.5]:
    mask = rng.rand(*Xb_te.shape) < rate
    X_missing = Xb_te.copy()
    X_missing[mask] = np.nan
    rows.append({"결측 비율": rate, "정확도": round(model.score(imputer.transform(X_missing), yb_te), 3)})
pd.DataFrame(rows)

## 4. 방향성 테스트

당뇨 데이터에서 **체질량지수(bmi)**가 높아지면 진행도 예측도 높아지는 것이 상식입니다. 모든 환자의 bmi만 조금 올렸을 때 예측이 올라가는지 확인합니다.

In [ ]:
d = load_diabetes()
Xd, yd = d.data, d.target
bmi = list(d.feature_names).index("bmi")
reg = LinearRegression().fit(Xd, yd)

X_up = Xd.copy()
X_up[:, bmi] += 0.01            # 이 데이터는 이미 표준화되어 있어 작은 값을 더합니다
increased = (reg.predict(X_up) > reg.predict(Xd)).mean()
print(f"bmi를 올렸을 때 예측이 올라간 환자 비율: {increased:.0%}")

- 방향성 테스트는 **정답 라벨 없이도** 모델의 상식적인 동작을 확인할 수 있습니다.
- 2일차에 이런 테스트를 PyTest 코드로 작성해 자동으로 실행합니다.